# 2. 图状态（State）管理

## 2.1 状态定义

状态的定义实际上是在声明状态的Schema，后者是状态字段的完整描述。

官方推荐了三种Schema的方式：`TypedDict`、`dataclass`、`Pydanic`

### 2.1.1 `TypedDict`

In [9]:
from dataclasses import dataclass
from typing import TypedDict, Annotated
from operator import add

from langgraph.graph import START, END, StateGraph
from pydantic import BaseModel


class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state["cur_id"]
    return {
        "logs": ["node1 运行完成"],
        "cur_id": pre_id + " node_1 "
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state["cur_id"]
    return {
        "logs": ["node2 运行完成"],
        "cur_id": pre_id + " node_2 "
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()

result = graph.invoke({"cur_id": "start"})
print(result)

{'logs': ['node1 运行完成', 'node2 运行完成'], 'cur_id': 'start node_1  node_2 '}


### 2.1.2 `dataclass`

In [7]:
@dataclass
class OverAllState:
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        "logs": ["node_1 运行完成"],
        "cur_id": pre_id + " node_1 "
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return OverAllState(
        logs = state.logs + ["node_2 运行完成"],
        cur_id = pre_id + " node_2 "
    )

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()

result = graph.invoke({"cur_id": "start"})
print(result)

{'logs': ['node_1 运行完成', 'node_1 运行完成', 'node_2 运行完成'], 'cur_id': 'start node_1  node_2 '}


### 2.1.3 `Pydanic`

In [10]:
class OverAllState(BaseModel):
    logs: Annotated[list[str], add]
    cur_id: str

def note_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        "logs": ["node_1 运行完成"],
        "cur_id": pre_id + " node_1 "
    }

def note_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return OverAllState(
        logs = state.logs + ["node_2 运行完成"],
        cur_id = pre_id + " node_2 "
    )

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()

result = graph.invoke({"cur_id": "start"})
print(result)

{'logs': ['node1 运行完成', 'node2 运行完成'], 'cur_id': 'start node_1  node_2 '}


### 2.1.4 校验行为

学习LangChain的结构化输出时我们提到：Pydantic对格式要求最严格，如果模型返回的内容不符合结构化Schema的要求，则抛出ValidationError。而其余方式都不会对模型的返回结果进行校验，即便模型返回的内容不符合结构化要求，也会原样返回给用户。

而作为LangGraph计算图的状态时，这三种方式都要求字段名称完全一致。只是处理方式不同。具体规则如下

**输入字段不匹配**

1. TypedDict
`TypedDict`将输入字段视为字典的Key，不匹配时抛出KeyError异常。

2. dataclass
`dataclass`将输入字段视为类的属性，不匹配时抛出TypeError（类型错误）异常。

3. Pydantic
`Pydantic`对输入字段进行校验，不匹配时抛出ValidationError异常。

**节点返回字段不匹配**

图节点返回的是对于状态的更新，如果返回字段和状态字段不匹配，上述三种Schema定义方式的行为是统一的：状态更新会被忽略。

### 2.1.5. 推荐用法
在实际使用中，推荐优先使用 `TypedDict` 定义 LangGraph 状态图的 State Schema。

大多数官方案例也采用 `TypedDict` 方式定义状态 Schema。这种方式写法简洁、结构清晰，能够直接描述状态中包含哪些字段，以及每个字段对应的数据类型，非常适合用于定义图运行过程中的共享状态。

相比普通 `dict`，`TypedDict` 可以提供更明确的字段约束和类型提示；相比 `dataclass`，`TypedDict` 更贴近 LangGraph 中状态的更新方式，因为节点通常返回的是表示“部分状态更新”的字典，而不是完整对象；相比 `Pydantic BaseModel`，它又更加轻量，不会引入额外的数据校验开销。因此，在没有复杂校验需求的情况下，`TypedDict` 是定义 LangGraph State Schema 的首选方式。